# Huấn luyện Mô hình Chẩn đoán DR (DenseNet-121, 5 mức)

Notebook này được thiết kế để chạy trên **Kaggle** (khuyên dùng) hoặc **Google Colab**.

- Dataset: **APTOS 2019 Blindness Detection**
- Mục tiêu: phân loại **5 mức độ** (0–4)
- Xuất model cho backend: `dr_classifier.h5` (1 file)

In [ ]:
# Cài đặt thư viện cần thiết (chạy trên Colab/Kaggle)
!pip install -q tensorflow scikit-learn numpy pandas opencv-python-headless matplotlib seaborn joblib

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score, roc_auc_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import layers
from tensorflow.keras.models import Model

# Cấu hình (APTOS 2019)
NUM_CLASSES = 5
IMG_SIZE = 224  # DenseNet121 input size
BATCH_SIZE = 16  # Batch nhỏ → gradient mượt hơn, converge tốt hơn
SEED = 42

# Kaggle path (competitions mount)
DATA_DIR = "/kaggle/input/competitions/aptos2019-blindness-detection"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")

assert os.path.exists(TRAIN_CSV), f"Không tìm thấy: {TRAIN_CSV}"
print("Dataset OK:", DATA_DIR)

tf.random.set_seed(SEED)
np.random.seed(SEED)


## 2. Tiền xử lý ảnh (CLAHE + Crop)

In [ ]:
def _crop_black_borders(img_bgr: np.ndarray, threshold: int = 10) -> np.ndarray:
    """Crop viền đen quanh vùng fundus (nhanh, đủ tốt cho APTOS)."""
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = (gray > threshold).astype(np.uint8) * 255
    coords = cv2.findNonZero(mask)
    if coords is None:
        return img_bgr

    x, y, w, h = cv2.boundingRect(coords)
    if w <= 0 or h <= 0:
        return img_bgr
    return img_bgr[y : y + h, x : x + w]


def _apply_clahe_lab(img_bgr: np.ndarray, clip_limit: float = 2.0, tile_grid_size=(8, 8)) -> np.ndarray:
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


def preprocess_image_densenet(image_path: str, img_size: int = IMG_SIZE) -> np.ndarray:
    """Read -> crop -> resize -> CLAHE -> DenseNet preprocess.

    Returns float32 (H,W,3) RGB after preprocess_input.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    img = _crop_black_borders(img)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    img = _apply_clahe_lab(img)

    # Keras applications expect RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    x = preprocess_input(img_rgb.astype(np.float32))
    return x


## 3. Chuẩn bị dữ liệu (train/val)

In [ ]:
# Load train.csv

df = pd.read_csv(TRAIN_CSV)
# df columns: id_code, diagnosis

df["image_path"] = df["id_code"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, f"{x}.png"))

# Stratified split: train/val/test
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["diagnosis"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["diagnosis"],
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Class distribution (train):")
print(train_df["diagnosis"].value_counts().sort_index())


## 4. Data Generator + Class Weight

In [ ]:
def _augment(img: np.ndarray) -> np.ndarray:
    """Augmentation nhẹ cho ảnh fundus: flip + xoay nhỏ + brightness."""
    if np.random.rand() < 0.5:
        img = cv2.flip(img, 1)   # horizontal flip
    if np.random.rand() < 0.5:
        img = cv2.flip(img, 0)   # vertical flip
    angle = np.random.uniform(-15, 15)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REFLECT_101)
    factor = np.random.uniform(0.9, 1.1)
    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return img


class APTOSSequence(tf.keras.utils.Sequence):
    def __init__(self, df: pd.DataFrame, batch_size: int, shuffle: bool = True,
                 augment: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, idx: int):
        batch_indexes = self.indexes[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indexes]

        x_batch = np.zeros((len(batch_df), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y_batch = batch_df["diagnosis"].values.astype(np.int32)

        for i, path in enumerate(batch_df["image_path"].values):
            img = cv2.imread(path)
            if img is None:
                x_batch[i] = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
                continue
            img = _crop_black_borders(img)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            if self.augment:
                img = _augment(img)
            img = _apply_clahe_lab(img)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            x_batch[i] = preprocess_input(img_rgb.astype(np.float32))

        return x_batch, y_batch


# augment=True chỉ cho tập train
train_gen = APTOSSequence(train_df, batch_size=BATCH_SIZE, shuffle=True,  augment=True)
val_gen   = APTOSSequence(val_df,   batch_size=BATCH_SIZE, shuffle=False, augment=False)
test_gen  = APTOSSequence(test_df,  batch_size=BATCH_SIZE, shuffle=False, augment=False)

# Class weights (giảm ảnh hưởng mất cân bằng)
classes = np.arange(NUM_CLASSES)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values,
)
class_weight = {i: float(w) for i, w in enumerate(class_weights)}
print("class_weight:", class_weight)


## 5. Xây dựng DenseNet-121 (5 lớp) + Metric QWK

In [ ]:
def build_densenet121_classifier(num_classes: int = NUM_CLASSES) -> tf.keras.Model:
    backbone = DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg",
        name="densenet121_backbone",
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


class QWKCallback(tf.keras.callbacks.Callback):
    """Chạy validation 1 lần/epoch: tính val_loss, val_accuracy, val_qwk.
    Dùng thay cho validation_data trong model.fit() để tránh chạy val 2 lần.
    """

    def __init__(self, val_sequence: tf.keras.utils.Sequence):
        super().__init__()
        self.val_sequence = val_sequence
        self.best_qwk = -1.0
        self.loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_true_list, y_pred_list, loss_acc = [], [], []

        for i in range(len(self.val_sequence)):
            x_batch, y_batch = self.val_sequence[i]
            proba = self.model(tf.constant(x_batch), training=False).numpy()
            pred = np.argmax(proba, axis=1)
            batch_loss = self.loss_fn(y_batch, proba).numpy()
            loss_acc.append((batch_loss, len(y_batch)))
            y_true_list.append(y_batch)
            y_pred_list.append(pred)

        y_true = np.concatenate(y_true_list)
        y_pred = np.concatenate(y_pred_list)
        total = len(y_true)

        val_loss = sum(l * n for l, n in loss_acc) / total
        val_acc  = np.mean(y_true == y_pred)
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

        logs["val_loss"]     = float(val_loss)
        logs["val_accuracy"] = float(val_acc)
        logs["val_qwk"]      = float(qwk)
        print(f"\nval_loss: {val_loss:.5f} — val_acc: {val_acc:.5f} — val_qwk: {qwk:.5f}")

        if qwk > self.best_qwk:
            self.best_qwk = qwk
            self.model.save("dr_classifier_best_qwk.h5")


model = build_densenet121_classifier(NUM_CLASSES)
model.summary()

qwk_cb = QWKCallback(val_gen)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, verbose=1, min_lr=1e-6
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
)


## 6. Huấn luyện (2 giai đoạn) + Lưu model cho backend

Sau khi chạy xong, bạn sẽ có file `dr_classifier.h5`.

- Trên Kaggle: dùng menu **Output** để download file.
- Ở máy local: copy `dr_classifier.h5` vào `DR_Diagnosis_System/backend/models/` rồi chạy backend.

In [ ]:
# Giai đoạn 1: train head (freeze backbone) — để head converge tốt trước
# Không dùng validation_data ở đây — QWKCallback đã lo validation 1 lần/epoch
history1 = model.fit(
    train_gen,
    epochs=15,
    class_weight=class_weight,
    callbacks=[qwk_cb, reduce_lr, early_stop],
)

# Giai đoạn 2: fine-tune sâu hơn
backbone = model.get_layer("densenet121_backbone")
backbone.trainable = True

# Mở 100 layer cuối để học feature đặc trưng hơn
for layer in backbone.layers[:-100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),  # LR nhỏ để fine-tune ổn định
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

# Callback riêng cho giai đoạn 2 với patience dài hơn
early_stop2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
)
reduce_lr2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, verbose=1, min_lr=1e-7
)

history2 = model.fit(
    train_gen,
    epochs=25,
    class_weight=class_weight,
    callbacks=[qwk_cb, reduce_lr2, early_stop2],
)

# Lưu model cuối (backend sẽ dùng file này)
model.save("dr_classifier.h5")
print("Saved: dr_classifier.h5")
print("Also saved best by QWK (if improved): dr_classifier_best_qwk.h5")


In [ ]:
# Đánh giá cuối cùng trên tập test (QWK, AUC, confusion matrix)
# Dùng model.predict() với steps cố định — tránh lặp vô tận qua Sequence
y_proba = model.predict(test_gen, steps=len(test_gen), verbose=1)
y_pred = np.argmax(y_proba, axis=1)

# Lấy nhãn thật theo đúng thứ tự (test_gen shuffle=False nên indexes không đổi)
y_true = test_gen.df["diagnosis"].iloc[test_gen.indexes].values

test_qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
test_acc = np.mean(y_true == y_pred)
print(f"Test Accuracy: {test_acc:.5f}")
print(f"Test QWK: {test_qwk:.5f}")

# AUC cho đa lớp (one-vs-rest)
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=NUM_CLASSES)
auc_macro = roc_auc_score(y_true_onehot, y_proba, average="macro", multi_class="ovr")
auc_micro = roc_auc_score(y_true_onehot, y_proba, average="micro", multi_class="ovr")
print(f"Test AUC (macro-ovr): {auc_macro:.5f}")
print(f"Test AUC (micro-ovr): {auc_micro:.5f}")

print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


## 7. Visualization - Confusion Matrix & Training History

In [ ]:
import seaborn as sns

# 1. Confusion Matrix Heatmap
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Vẽ heatmap với giá trị số và phần trăm
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Level 0', 'Level 1', 'Level 2', 'Level 3', 'Level 4'],
            yticklabels=['Level 0', 'Level 1', 'Level 2', 'Level 3', 'Level 4'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - DR Classification (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Normalized Confusion Matrix (% per class)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='RdYlGn', vmin=0, vmax=1,
            xticklabels=['Level 0', 'Level 1', 'Level 2', 'Level 3', 'Level 4'],
            yticklabels=['Level 0', 'Level 1', 'Level 2', 'Level 3', 'Level 4'],
            cbar_kws={'label': 'Recall (Sensitivity)'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Normalized Confusion Matrix (Recall per Class)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Training History - Loss & Accuracy
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Combine history từ 2 giai đoạn
all_loss = history1.history.get('loss', []) + history2.history.get('loss', [])
all_acc = history1.history.get('accuracy', []) + history2.history.get('accuracy', [])

# Loss
axes[0].plot(all_loss, label='Training Loss', linewidth=2, color='#E74C3C')
axes[0].axvline(x=len(history1.history.get('loss', [])), color='gray', linestyle='--', 
                label='Fine-tune Start', alpha=0.7)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(all_acc, label='Training Accuracy', linewidth=2, color='#27AE60')
axes[1].axvline(x=len(history1.history.get('accuracy', [])), color='gray', linestyle='--',
                label='Fine-tune Start', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy Over Time', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: confusion_matrix.png, confusion_matrix_normalized.png, training_history.png")

## 8. Sensitivity & Specificity (Metrics Y tế)

In [ ]:
from sklearn.metrics import recall_score, precision_score

# Sensitivity (Recall) per class - tỷ lệ phát hiện đúng bệnh
sensitivity = recall_score(y_true, y_pred, average=None)

# Precision per class - độ chính xác khi dự đoán
precision = precision_score(y_true, y_pred, average=None, zero_division=0)

# Specificity per class - tỷ lệ phát hiện đúng không bệnh
specificity = []
for i in range(NUM_CLASSES):
    tn = np.sum((y_true != i) & (y_pred != i))
    fp = np.sum((y_true != i) & (y_pred == i))
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificity.append(spec)

# Tạo DataFrame để hiển thị đẹp
metrics_df = pd.DataFrame({
    'Class': [f'Level {i}' for i in range(NUM_CLASSES)],
    'Sensitivity (Recall)': [f'{s:.4f}' for s in sensitivity],
    'Specificity': [f'{s:.4f}' for s in specificity],
    'Precision': [f'{p:.4f}' for p in precision],
})

print("=" * 70)
print("METRICS Y TẾ - PHÂN TÍCH TỪNG MỨC ĐỘ DR")
print("=" * 70)
print(metrics_df.to_string(index=False))
print("=" * 70)

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(NUM_CLASSES)
width = 0.25

bars1 = ax.bar(x - width, sensitivity, width, label='Sensitivity', color='#3498DB')
bars2 = ax.bar(x, specificity, width, label='Specificity', color='#2ECC71')
bars3 = ax.bar(x + width, precision, width, label='Precision', color='#E67E22')

ax.set_xlabel('DR Level', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Medical Metrics per DR Level', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Level {i}' for i in range(NUM_CLASSES)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

# Thêm giá trị lên các bar
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('medical_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: medical_metrics.png")

## 9. ROC Curves (One-vs-Rest)

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(12, 8))

# Vẽ ROC curve cho từng class
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2.5,
             label=f'Level {i} (AUC = {roc_auc:.3f})')

# Đường random baseline
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier (AUC = 0.500)', alpha=0.5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curves - Multi-class DR Classification (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: roc_curves.png")

## 10. Lưu Kết quả & Phân tích Misclassified Examples

In [ ]:
# 1. Lưu predictions ra CSV
results_df = pd.DataFrame({
    'id_code': test_df['id_code'].values,
    'true_label': y_true,
    'predicted_label': y_pred,
    'correct': (y_true == y_pred).astype(int),
})

# Thêm xác suất cho từng class
for i in range(NUM_CLASSES):
    results_df[f'prob_level_{i}'] = y_proba[:, i]

results_df.to_csv('test_predictions.csv', index=False)
print("✅ Saved: test_predictions.csv")

# 2. Lưu metrics summary ra file text
with open('test_metrics_summary.txt', 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("TỔNG KẾT KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH DR CLASSIFICATION\n")
    f.write("=" * 70 + "\n\n")
    
    f.write(f"Dataset: APTOS 2019 Blindness Detection\n")
    f.write(f"Architecture: DenseNet-121 (Transfer Learning)\n")
    f.write(f"Test Set Size: {len(y_true)} images\n\n")
    
    f.write("-" * 70 + "\n")
    f.write("OVERALL METRICS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Test Accuracy:        {test_acc:.5f} ({test_acc*100:.2f}%)\n")
    f.write(f"Test QWK (Kappa):     {test_qwk:.5f}\n")
    f.write(f"Test AUC (macro-ovr): {auc_macro:.5f}\n")
    f.write(f"Test AUC (micro-ovr): {auc_micro:.5f}\n\n")
    
    f.write("-" * 70 + "\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("-" * 70 + "\n")
    f.write(classification_report(y_true, y_pred, 
                                  target_names=[f'Level {i}' for i in range(NUM_CLASSES)],
                                  digits=4))
    f.write("\n")
    
    f.write("-" * 70 + "\n")
    f.write("MEDICAL METRICS PER CLASS\n")
    f.write("-" * 70 + "\n")
    f.write(metrics_df.to_string(index=False))
    f.write("\n\n")
    
    f.write("-" * 70 + "\n")
    f.write("CONFUSION MATRIX\n")
    f.write("-" * 70 + "\n")
    f.write(str(confusion_matrix(y_true, y_pred)))
    f.write("\n")

print("✅ Saved: test_metrics_summary.txt")

# 3. Phân tích misclassified examples
misclassified_mask = (y_true != y_pred)
misclassified_df = test_df.iloc[np.where(misclassified_mask)[0]].copy()
misclassified_df['true_label'] = y_true[misclassified_mask]
misclassified_df['predicted_label'] = y_pred[misclassified_mask]
misclassified_df['error_magnitude'] = np.abs(
    y_true[misclassified_mask] - y_pred[misclassified_mask]
)

print("\n" + "=" * 70)
print(f"MISCLASSIFIED EXAMPLES: {len(misclassified_df)} / {len(test_df)} ({len(misclassified_df)/len(test_df)*100:.2f}%)")
print("=" * 70)
print("\nTop 10 Misclassified (by error magnitude):")
print(misclassified_df.nlargest(10, 'error_magnitude')[['id_code', 'diagnosis', 'true_label', 'predicted_label', 'error_magnitude']])

# Lưu danh sách misclassified
misclassified_df.to_csv('misclassified_examples.csv', index=False)
print("\n✅ Saved: misclassified_examples.csv")

# 4. Confusion pattern analysis
print("\n" + "=" * 70)
print("CONFUSION PATTERN ANALYSIS")
print("=" * 70)
for true_class in range(NUM_CLASSES):
    mask = (y_true == true_class) & (y_pred != true_class)
    if np.sum(mask) > 0:
        confused_with = y_pred[mask]
        from collections import Counter
        confusion_counts = Counter(confused_with)
        print(f"\nLevel {true_class} most confused with:")
        for pred_class, count in confusion_counts.most_common(3):
            print(f"  → Level {pred_class}: {count} times ({count/np.sum(mask)*100:.1f}%)")

print("\n" + "=" * 70)
print("✅ ALL RESULTS SAVED SUCCESSFULLY!")
print("=" * 70)
print("\nFiles generated:")
print("  - dr_classifier.h5 (model cho backend)")
print("  - dr_classifier_best_qwk.h5 (best model by QWK)")
print("  - test_predictions.csv")
print("  - test_metrics_summary.txt")
print("  - misclassified_examples.csv")
print("  - confusion_matrix.png")
print("  - confusion_matrix_normalized.png")
print("  - training_history.png")
print("  - medical_metrics.png")
print("  - roc_curves.png")

## 11. Download Files & Next Steps

### 📥 Download các file (trên Kaggle/Colab):

**Model files:**
- `dr_classifier.h5` - Model chính cho backend
- `dr_classifier_best_qwk.h5` - Best model theo QWK

**Results & Reports:**
- `test_predictions.csv` - Chi tiết predictions từng ảnh
- `test_metrics_summary.txt` - Tổng hợp metrics đầy đủ
- `misclassified_examples.csv` - Danh sách ảnh bị phân loại sai

**Visualizations (PNG):**
- `confusion_matrix.png` & `confusion_matrix_normalized.png`
- `training_history.png`
- `medical_metrics.png`
- `roc_curves.png`

### 📂 Cách sử dụng model cho backend:

```bash
# 1. Download dr_classifier.h5 từ Kaggle Output
# 2. Copy vào thư mục backend:
cp dr_classifier.h5 DR_Diagnosis_System/backend/models/

# 3. Restart backend Flask app
cd DR_Diagnosis_System/backend
python app.py
```

### 📊 Sử dụng kết quả cho báo cáo:

1. **test_metrics_summary.txt** - Dùng cho phần kết quả đánh giá
2. **PNG files** - Chèn vào Word/LaTeX cho luận văn
3. **misclassified_examples.csv** - Phân tích lỗi và thảo luận

### ⚡ Tips:
- Nếu QWK < 0.7: cần train thêm hoặc điều chỉnh hyperparameters
- Nếu một class có sensitivity thấp: thêm augmentation cho class đó
- So sánh `dr_classifier.h5` vs `dr_classifier_best_qwk.h5` để chọn model tốt hơn